# Data Exploration

In [42]:
# Load the ratings using the function from data.py
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from data import load_movielens, split_by_time, to_tensors, seen_matrix
from model import build_graph, BehavioralRecommender
from metrics import evaluate, calibration_error

In [30]:
df = load_movielens()   # downloads into data/ the first time
df.head(10)

,user,item,rating,time
0,6039,802,4,956703932
1,6039,579,5,956703954
2,6039,2191,4,956703954
3,6039,1781,4,956703977
4,6039,1839,5,956703977
5,6039,559,4,956704056
6,6039,1316,3,956704056
7,6039,207,5,956704056
8,6039,2895,5,956704056
9,6039,3272,4,956704056


In [31]:
# Overview: size, column types, and basic statistics
print(f"{len(df):,} ratings | {df.user.nunique():,} users | {df.item.nunique():,} movies")
df["date"] = pd.to_datetime(df["time"], unit="s")
print("From", df.date.min().date(), "to", df.date.max().date())
df.describe()

1,000,209 ratings | 6,040 users | 3,706 movies
From 2000-04-25 to 2003-02-28


,user,item,rating,time,date
count,1.000209e+06,1.000209e+06,1.000209e+06,1.000209e+06,1000209
mean,3.023512e+03,1.730481e+03,3.581564e+00,9.722437e+08,2000-10-22 19:41:35.404665
min,0.000000e+00,0.000000e+00,1.000000e+00,9.567039e+08,2000-04-25 23:05:32
25%,1.505000e+03,9.650000e+02,3.000000e+00,9.653026e+08,2000-08-03 11:37:17
50%,3.069000e+03,1.657000e+03,4.000000e+00,9.730180e+08,2000-10-31 18:46:46
75%,4.475000e+03,2.565000e+03,4.000000e+00,9.752209e+08,2000-11-26 06:42:19
max,6.039000e+03,3.705000e+03,5.000000e+00,1.046455e+09,2003-02-28 17:49:50
std,1.728413e+03,1.017026e+03,1.117102e+00,1.215256e+07,NaN


In [43]:
# How active are users, how popular are movies, and what does the time split look like?
per_user = df.groupby("user").size()
per_movie = df.groupby("item").size()
train, val, test = split_by_time(df)
n_users, n_items = df.user.max() + 1, df.item.max() + 1
pd.DataFrame({
    "ratings per user": per_user.describe(),
    "ratings per movie": per_movie.describe(),
}).round(1), pd.DataFrame({
    "n_events": [len(train), len(val), len(test)],
    "n_users": [train.user.nunique(), val.user.nunique(), test.user.nunique()],
    "first date": [pd.to_datetime(s.time.min(), unit="s").date() for s in (train, val, test)],
    "last date": [pd.to_datetime(s.time.max(), unit="s").date() for s in (train, val, test)],
}, index=["train", "val", "test"])

(       ratings per user  ratings per movie
 count            6040.0             3706.0
 mean              165.6              269.9
 std               192.7              384.0
 min                20.0                1.0
 25%                44.0               33.0
 50%                96.0              123.5
 75%               208.0              350.0
 max              2314.0             3428.0,
        n_events  n_users  first date   last date
 train    800167     5400  2000-04-25  2000-12-02
 val       23841      523  2000-12-02  2000-12-29
 test      80608     1008  2000-12-30  2003-02-28)

In [44]:
train[['user', 'item']].values

array([[6039,  802],
       [6039,  579],
       [6039, 2191],
       ...,
       [1874, 1884],
       [1874,  426],
       [1874, 1574]], shape=(800167, 2))

# Model

In [45]:
pairs = np.unique(train[['user', 'item']].values, axis=0) # user-item pair is unique
u, i = pairs[:, 0], pairs[:, 1] + n_users
rows = np.concatenate([u, i])   # both directions: the graph is undirected
cols = np.concatenate([i, u])
degree = np.bincount(rows, minlength=n_users + n_items)
values = 1.0 / np.sqrt(degree[rows] * degree[cols])
values

array([0.00422864, 0.00379995, 0.00414891, ..., 0.00276609, 0.0019418 ,
       0.00403653], shape=(1600334,))

In [46]:
graph = build_graph(train, n_users, n_items)
model = BehavioralRecommender(n_users, n_items, dim=128, graph=graph, rationality='global')
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

In [ ]:
best_loglik, best_epoch, history = -float("inf"), 0, []
epochs = 1000
batch_size = 1024
weight_decay = 1e-5
rationality='global'
patience = 5

train_users, train_items = to_tensors(train)
val_users, val_items = to_tensors(val)
test_users, test_items = to_tensors(test)
seen_before_val = seen_matrix(train, n_users, n_items)
seen_before_test = seen_matrix(pd.concat([train, val]), n_users, n_items)


for epoch in range(1, epochs + 1):
    model.train()
    order = torch.randperm(len(train_users))
    total_loss = 0.0
    for start in range(0, len(order), batch_size):
        batch = order[start:start + batch_size]
        users, items = train_users[batch], train_items[batch]

        nll = F.cross_entropy(model.logits(users), items)          # - mean log P(chosen item)
        prior = model.prior_penalty() / len(train_users)
        l2 = weight_decay * model.embedding.weight[users].pow(2).sum(dim=1).mean()
        loss = nll + prior + l2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += nll.item() * len(batch)

    val_metrics = evaluate(model, val_users, val_items, seen_before_val)
    row = {"epoch": epoch, "train_nll": total_loss / len(order), **{f"val_{k}": v for k, v in val_metrics.items()}}
    if rationality == "global":
        row["lambda"] = model.log_lambda.exp().item()
    history.append(row)
    print("  ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}" for k, v in row.items()))

    # Keep the epoch with the best validation log-likelihood; stop if no progress.
    if val_metrics["loglik"] > best_loglik:
        best_loglik, best_epoch = val_metrics["loglik"], epoch
    elif epoch - best_epoch >= patience:
        print(f"No improvement for {patience} epochs, stopping.")
        break
